Goal for this file: test out widget functions of shiny to build a graph with a dropdown menu for selecting the year

In [ ]:
# import packages
# Import packages
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import platform
from pathlib import Path
import geopandas as gpd
import calendar
import pygris



In [ ]:
# set seaborne theme
sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette='magma',
    font="Calibri",
    rc={
        # Figure
        "figure.figsize": (10, 6),
        "figure.dpi": 120,

        # Background
        "axes.facecolor": "white",
        "figure.facecolor": "white",

        # Grid
        "axes.grid": True,
        "grid.color": "#E6E9EC",
        "grid.linestyle": "-",
        "grid.linewidth": 0.7,

        # Axes
        "axes.edgecolor": "#D5DADF",
        "axes.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,

        # Ticks
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,

        # Legend
        "legend.frameon": True,
        "legend.fontsize": 7,
        "legend.loc": "upper right",
        "legend.framealpha": 0.9,
        "legend.title_fontsize": 9,
        "legend.edgecolor": "#D5DADF",
        "legend.facecolor": "white",

        # Avoid scientific notation and offset notation
        "axes.formatter.useoffset": False,
        "axes.formatter.limits": (-100, 100),
    }
)

# Set default color cycle
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=sns.color_palette('magma')
)

In [6]:
# access fire data via function
def read_fires(year):
    #  SET PATH TO DATA FOLDER ON BOX DRIVE
    if platform.system() == 'Windows':
        box_path = Path.home() / 'Box'
    # if mac:
    else:
        box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

    #Confirm that the box path is valid
    if box_path.exists():
        print(f'Box folder found at {box_path}')
    else:
        print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

    # create path object that goes to the specific folder on Box drive
    # uploading wetlands that are bigger than a hectare
    fire_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'fire25_1.gdb'
    print(fire_path.exists())

    fire_shp = gpd.read_file(fire_path, layer='firep25_1')

    # FILTER BY DESIRED YEAR
    fire_yr = fire_shp[fire_shp['YEAR_'] == year]

    # drop rows with null values in geometry column
    fire_yr = fire_yr.dropna(subset=['geometry'])

    # add column for month of alarm date
    # add a column for month of alarm date
    # add a column for month of alarm date
    fire_yr['Alarm Month'] = fire_yr['ALARM_DATE'].dt.month

    fire_yr['Alarm Month Abb'] = pd.Categorical(
        values = fire_yr['ALARM_DATE'].dt.month_name().str[:3],
        categories=calendar.month_abbr[1:]
    )

    return fire_yr

In [7]:
# access wetland data
# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")

Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


In [8]:
# DISTANCE TO NEAREST FIRE FUNCTION
def dist_fire(wetlands_shp, fire_shp):
    # convert to same CRS
    wetlands_shp = wetlands_shp.to_crs(epsg=26910)
    fire_shp = fire_shp.to_crs(epsg=26910)
    
    
    # Nearest Neighbor
    nearest_fires = gpd.sjoin_nearest(
        wetlands_shp, fire_shp, how='left', distance_col='Dist_Closest_m'
    )

    # convert distance column from meters to kilometers
    nearest_fires['Dist_Closest_km'] = nearest_fires['Dist_Closest_m'] / 1000

    # add information to a table called fire_stats
    fire_stats = (nearest_fires.groupby('C System')['Dist_Closest_km'].agg(
        Avg_Dist_Fire_km="mean",
        Std_Dist_Fire_km="std",
        Median_Dist_Fire_km="median",
        Range_Dist_Fire_km=lambda x: x.max() - x.min(),
        IQR_Dist_Fire_km=lambda x: x.quantile(0.75) - x.quantile(0.25),
        ).reset_index()
    )

    return nearest_fires, fire_stats
# outputs are a shapefile and the fire_stats table

In [21]:
# INTERSECT function
# inputs for this function are both shapefiles
# this function edited to create a separate stats table (that we can later join by wetland type if needed)
def overlap(wetlands_shp, fire_shp):

    # convert to same CRS
    wetlands_shp = wetlands_shp.to_crs(epsg=26910)
    fire_shp = fire_shp.to_crs(epsg=26910)
    
    # calculate the total area of fire in CA in hectares
    fire_area = fire_shp.geometry.area.sum() / 10000
    
    # calculate the total area of wetlands in CA
    wetlands_area = wetlands_shp.geometry.area.sum() / 10000

    print(f'AREA OF ALL FIRE: {fire_area} hectares')
    print(f'AREA OF ALL WETLANDS: {wetlands_area} hectares')
    
    # PERFORM SPATIAL JOIN
    intersect = gpd.overlay(wetlands_shp, fire_shp, how="intersection")

    # add a column to the intersected SHP that shows intersect area in hectares
    # 1 hectare = 10,000 square meters
    intersect['int_area_ha']=intersect.geometry.area / 10000

    # ADD PCT WETLAND BURNED BY SYSTEM
    # get wetland area by system
    total_wetlands = wetlands_shp.groupby('C System')['geometry'].apply(lambda x: x.area.sum() / 10000)
    

    # CREATE FIRE_STATS TABLE
    fire_stats = pd.DataFrame({
        'C System': total_wetlands.index
    })

    # ADD WETLAND AREA BURNED BY SYSTEM TO FIRE_STATS TABLE
    burned = intersect.groupby('C System')['int_area_ha'].sum()
    fire_stats['Burned Area (ha)'] = fire_stats['C System'].map(burned).fillna(0)

   
    # sanity check - total wetland area does match the sum of each system
    print(round(total_wetlands.sum()) == round(wetlands_area))

    # add wetland area to fire stats table
    fire_stats['Total Wetland Area (ha)'] = fire_stats['C System'].map(total_wetlands)

    fire_stats['Percent Wetland Area Burned'] = (fire_stats['Burned Area (ha)'] / fire_stats['Total Wetland Area (ha)']) *100

    # add pct fire that happened inside wetlands
    fire_stats['Percent of Fire Area in Wetlands'] = (fire_stats['Burned Area (ha)'] / fire_area) * 100


    return intersect, fire_stats

## We need to precalculate spatial data to make the dashboard run faster
### This code only needs to be run once - I will hashtag it out after

### Nearest Fire data

In [ ]:
# # i think we need to presave spatial data results for this to work
# # Years to process
# # 2020 is included, i just had to interrupt the run
# year_list = [2021, 2022, 2023, 2024, 2025]

# # Set path to Box folder based on operating system
# if platform.system() == 'Windows':
#     box_path = Path.home() / 'Box'

# # Mac
# else:
#     box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"


# # Confirm Box folder exists
# if box_path.exists():
#     print(f'Box folder found at: {box_path}')
# else:
#     raise FileNotFoundError(
#         'Failed to locate Box folder. '
#         'Is Box Drive running and are you logged in?'
#     )


# # Create folder in Box to save yearly spatial data
# output_folder = (
#     box_path / 'ENVIRON 712L-Students' /
#     'Data' / 'BunkerAndersonTseng' /
#     'fire_distance_data'
# )

# # Create folder if it does not already exist
# output_folder.mkdir(parents=True, exist_ok=True)

# print(f'\nSpatial data will be saved to: {output_folder}')


# # Calculate and save spatial results for each year
# for year in year_list:
#     print(f"processing {year}...")

#     # Load fire data for selected year
#     fire_yr = read_fires(year)

#     # Calculate nearest fire distances
#     nearest_fires, fire_stats = dist_fire(wetlands, fire_yr)

#     # Save spatial dataset
#     nearest_fires.to_file(
#         output_folder / f"nearest_fires_{year}.gpkg",
#         driver="GPKG"
#     )

#     # Save statistics table
#     fire_stats.to_csv(
#         output_folder / f"fire_stats_{year}.csv",
#         index=False
#     )

#     print(f"Finished processing {year}")

Box folder found at: /Users/katanderson/Library/CloudStorage/Box-Box

Spatial data will be saved to: /Users/katanderson/Library/CloudStorage/Box-Box/ENVIRON 712L-Students/Data/BunkerAndersonTseng/fire_distance_data
processing 2021...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


/Users/katanderson/miniforge3/envs/env700-shiny/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined. Further messages of this type will be suppressed.
  return ogr_read(
/Users/katanderson/miniforge3/envs/env700-shiny/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP. Further messages of this type will be suppressed.
  return ogr_read(


Finished processing 2021
processing 2022...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
Finished processing 2022
processing 2023...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
Finished processing 2023
processing 2024...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
Finished processing 2024
processing 2025...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
Finished processing 2025


### Fire Overlap Data

In [22]:
# i think we need to presave spatial data results for this to work
# Years to process
year_list = [2020, 2021, 2022, 2023, 2024, 2025]

# Set path to Box folder based on operating system
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'

# Mac
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"


# Confirm Box folder exists
if box_path.exists():
    print(f'Box folder found at: {box_path}')
else:
    raise FileNotFoundError(
        'Failed to locate Box folder. '
        'Is Box Drive running and are you logged in?'
    )


# Create folder in Box to save yearly spatial data
output_folder = (
    box_path / 'ENVIRON 712L-Students' /
    'Data' / 'BunkerAndersonTseng' /
    'fire_overlap_data'
)

# Create folder if it does not already exist
output_folder.mkdir(parents=True, exist_ok=True)

print(f'\nSpatial data will be saved to: {output_folder}')


# Calculate and save spatial results for each year
for year in year_list:
    print(f"processing {year}...")

    # Load fire data for selected year
    fire_yr = read_fires(year)

    # Calculate overlap
    overlap_fires, fire_stats = overlap(wetlands, fire_yr)

    # Save spatial dataset
    overlap_fires.to_file(
        output_folder / f"overlap_fires_{year}.gpkg",
        driver="GPKG"
    )

    # Save statistics table
    fire_stats.to_csv(
        output_folder / f"fire_ol_stats_{year}.csv",
        index=False
    )

    print(f"Finished processing {year}")

Box folder found at: /Users/katanderson/Library/CloudStorage/Box-Box

Spatial data will be saved to: /Users/katanderson/Library/CloudStorage/Box-Box/ENVIRON 712L-Students/Data/BunkerAndersonTseng/fire_overlap_data
processing 2020...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
AREA OF ALL FIRE: 1693233.8986090533 hectares
AREA OF ALL WETLANDS: 3382322.7053495385 hectares
True
Finished processing 2020
processing 2021...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
AREA OF ALL FIRE: 1016977.3564255907 hectares
AREA OF ALL WETLANDS: 3382322.7053495385 hectares
True
Finished processing 2021
processing 2022...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
AREA OF ALL FIRE: 130732.43837623403 hectares
AREA OF ALL WETLANDS: 3382322.7053495385 hectares
True
Finished processing 2022
processing 2023...
Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True
AREA OF ALL FIRE: 139581.9096262231 hect

## Now we can write a function to do the plot by grabbing the correct year's data

In [17]:
# function for graphing
# takes already processed spatial data and graphs it based on year

def plot_firedist(year):
    

    g = sns.displot(
    data = fire_data[year],
    x = 'Dist_Closest_km',
    height=3,
    aspect=2,
    color="orange"
    )

    g.set(
        title = "Distance from Wetlands to Nearest Fire (km)",
        xlabel="Distance (km)",
        ylabel="Count"
    );
    return g



In [ ]:
def plot_fireint(year):
    data = fire_data[year]
    

In [18]:
#Create a dropdown widget for years
year_list = [2020, 2021, 2022, 2023, 2024, 2025]
year_dropdown = widgets.Dropdown(
    options=year_list,
    value=year_list[0],
    description='Year:',
)

## And here, we can make the widget area using the plot
### Graph distance to nearest fire for each wetland, by year

In [19]:
# load all the fire data ahead of time and store in dictionary

# Set path to Box folder
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

# Path to saved spatial data
output_folder = (
    box_path / 'ENVIRON 712L-Students' /
    'Data' / 'BunkerAndersonTseng' /
    'fire_distance_data'
)
# go through list of years and load file, store in dictionary
fire_data = {}
year_list = [2020, 2021, 2022, 2023, 2024, 2025]
for year in year_list:
    data = gpd.read_file(
        output_folder / f'nearest_fires_{year}.gpkg'
    )
    fire_data[year] = data




# make the UI
# Create an output area for the plot
plot_output = widgets.Output()

# Function to update the plot
def update_data_or_plot(change=None):
    #Direct output to the plot_output widget
    with plot_output:
        # Clear the previous plot
        clear_output(wait=True)

        # Generate the new plot
        g = plot_firedist(year_dropdown.value)

        # Explicitly display the plot
        display(g.figure)
        plt.close(g.figure)

# Attach observers
year_dropdown.observe(update_data_or_plot, names='value')

# Display the widgets and plot together
display(
    widgets.VBox([
        year_dropdown,
        plot_output
    ])
)

# Generate the initial plot
update_data_or_plot()


### Graph overlap in wetland and fire area